# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amarkosovac11/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


### Selected Lane: Refresh / Content Opportunity Scoring

**ML Task Type:** Ranking and Scoring

My goal is to develop a system that ranks web pages according to their potential value for a content-refresh review.

The unit of analysis is one web page.

The system would use available historical signals such as content age, days since the last update, impressions, CTR, and average search position.

The output would be a ranked list of pages, with higher scores representing higher estimated priority for manual review.

An SEO specialist would review the highest-ranked pages and decide which ones should actually be updated.

This is primarily a ranking problem because the practical decision is which pages to inspect first when time and resources are limited.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


### Proposed Target

The ideal target is an observed future content-refresh opportunity, measured using search-performance outcomes from a later time period.

For example, a future evaluation could investigate whether pages identified as high priority subsequently show measurable improvement after a documented refresh.

However, improvement after an update is not automatically proof that the update caused the improvement.

### Current Starter Dataset

The starter dataset contains historical page-level signals and derived indicators.

The existing `is_declining_label` may be useful for exploratory analysis, but it is derived from historical trend information and does not measure the future benefit of refreshing a page.

I will not treat a manually defined priority rule as independent ground truth.

### Proposed Modeling Approach

Initially, I will investigate descriptive scoring and baseline ranking methods.

Later, if appropriate historical and future observation windows are available, I will evaluate whether an ML model can predict a genuinely observed future outcome.

Features must be available before the prediction point to avoid data leakage.


## 3. Success metric

*One metric you can defend. What number means 'good'?*


### Primary Success Metric: Precision@10

The proposed primary evaluation metric is Precision@10.

This measures how many of the ten highest-ranked pages match a clearly defined, independently observed positive outcome.

For example, if 7 out of the top 10 pages have a valid positive outcome, Precision@10 would be 0.70.

### Baseline Comparison

I will compare the ML ranking with a simple rule-based baseline using the same evaluation data.

A useful ML model should outperform the baseline on held-out data, not only on training data.

### Practical Interpretation

Higher Precision@10 would mean fewer irrelevant pages are sent to SEO specialists for review.

However, this metric can only be calculated meaningfully once a suitable observed target is available.

### Initial Success Criterion

My initial goal is to demonstrate improvement over a predefined rule-based baseline on held-out data.

I will not claim that the model creates more organic traffic without appropriate real-world validation.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:

# ML-03: Frame the content-refresh ranking task

import os
from pathlib import Path
import pandas as pd
from IPython.display import display

# Load starter repository if necessary
repo = Path("/content/flyrank-ml-internship")

if not repo.exists():
    !git clone https://github.com/amarkosovac11/flyrank-ml-internship.git /content/flyrank-ml-internship

data_path = repo / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)

# The unit of analysis is one anonymized content item / web page
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

required = ["content_id"] + features
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

page_df = df[required].copy()

print("ML-03: Content Refresh Ranking")
print("Unit of analysis: one anonymized web page")
print("Total rows:", len(page_df))
print("Unique content IDs:", page_df["content_id"].nunique())
print("\nSample of real page-level data:")
display(page_df.head(10))

# Sketch of a proposed FUTURE outcome target
# Not available in the starter snapshot.
page_df["future_positive_outcome"] = pd.Series(
    pd.NA, index=page_df.index, dtype="boolean"
)

print("\nProposed target structure:")
display(page_df[[
    "content_id",
    "content_age_days",
    "impressions_90d",
    "future_positive_outcome"
]].head(10))

print("\nTarget status:")
print("Future outcomes are NOT available in this snapshot.")
print("The target is a schema placeholder, not a trained label.")

# Historical proxy inspection - NOT future ground truth
if "is_declining_label" in df.columns:
    print("\nHistorical exploratory proxy:")
    print(df["is_declining_label"].value_counts(dropna=False))
    print(
        "Warning: is_declining_label is derived from trend data "
        "and does not measure future refresh benefit."
    )

print("\nFeature summary:")
display(page_df[features].describe().round(2))

print("\nConclusion:")
print(
    "The dataset contains page-level features suitable for "
    "exploring content-refresh prioritization. "
    "A valid future outcome is needed for predictive evaluation."
)


Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 38), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 11.00 MiB/s, done.
Resolving deltas: 100% (38/38), done.
ML-03: Content Refresh Ranking
Unit of analysis: one anonymized web page
Total rows: 30000
Unique content IDs: 30000

Sample of real page-level data:


,content_id,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position
0,content_304f48230142,187,20,3803,0.76,10.6
1,content_a1fb4e703a9e,445,25,15320,0.05,20.3
2,content_9aa793d4d895,141,20,12581,0.09,36.5
3,content_331d6c4de07b,463,22,11751,0.49,6.2
4,content_d99b7a2d90ca,263,14,19140,0.13,44.0
5,content_d4084a4bc775,147,20,3970,0.03,8.5
6,content_9a34b442b552,90,20,20,0.00,7.0
7,content_a63219c6e95a,445,22,1724,0.06,21.2
8,content_5e6c160719bc,90,20,32574,0.09,46.0
9,content_c27558df2b0c,257,104,1240,0.16,4.9



Proposed target structure:


,content_id,content_age_days,impressions_90d,future_positive_outcome
0,content_304f48230142,187,3803,<NA>
1,content_a1fb4e703a9e,445,15320,<NA>
2,content_9aa793d4d895,141,12581,<NA>
3,content_331d6c4de07b,463,11751,<NA>
4,content_d99b7a2d90ca,263,19140,<NA>
5,content_d4084a4bc775,147,3970,<NA>
6,content_9a34b442b552,90,20,<NA>
7,content_a63219c6e95a,445,1724,<NA>
8,content_5e6c160719bc,90,32574,<NA>
9,content_c27558df2b0c,257,1240,<NA>



Target status:
Future outcomes are NOT available in this snapshot.
The target is a schema placeholder, not a trained label.

Feature summary:


,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position
count,30000.00,30000.00,30000.00,30000.00,30000.00
mean,256.17,46.10,5200.37,0.51,16.34
std,132.71,42.08,16838.02,3.28,15.22
min,90.00,1.00,1.00,0.00,0.00
25%,132.00,20.00,81.00,0.00,6.20
50%,236.00,20.00,731.00,0.07,10.80
75%,333.00,104.00,3615.25,0.29,22.30
max,564.00,373.00,517715.00,100.00,245.00



Conclusion:
The dataset contains page-level features suitable for exploring content-refresh prioritization. A valid future outcome is needed for predictive evaluation.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


### Why Machine Learning May Be Better Than a Fixed Rule

A basic rule-based approach might prioritize pages that have not been updated for more than 180 days.

However, content age alone does not necessarily indicate that a page needs improvement.

A newer page may have poor search performance, while an older page may continue to perform well.

The relationship between content age, impressions, CTR, and search position may be complex.

Machine learning could potentially identify combinations of signals that fixed thresholds overlook.

### Why a Baseline Is Still Important

Machine learning is not automatically better than a simple rule.

I will first create a transparent rule-based baseline and evaluate its usefulness.

Only if a machine learning model demonstrates better performance on appropriate held-out data would I consider it a meaningful improvement.

### Practical Business Action

The final system should help SEO specialists prioritize manual content reviews.

It should provide decision support, not automatically change pages or promise better Google rankings.

### Limitation

The starter dataset alone cannot prove that a content update causes future performance improvements. Any predictive model must be evaluated against an appropriate observed outcome.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.